# RQ2 Monotonicity Audit and Design

This notebook audits and designs the monotonicity pseudo-label component for **RQ2**.

## RQ2
To what extent can monotonicity pseudo-labels enforce expected monotone relationships between selected weather variables and predictions, and how does this influence accuracy?

## Purpose of this notebook
This notebook does **not** train monotonic models yet. Its purpose is to decide:

1. which monotonicity assumptions are defensible in the Nuuksio weather task,
2. whether those assumptions should be treated as global or local,
3. how monotonic comparison pairs should be constructed,
4. how monotonicity should be evaluated on labels and on the baseline model,
5. which final monotonicity specification should be used in the later training notebook.

## Deliverables
By the end of this notebook, the goal is to produce:

- a candidate monotonicity table,
- a preferred pair-construction rule,
- label-level monotonicity audit tables,
- baseline-model monotonicity audit tables,
- gradient-sign diagnostics,
- local counterfactual sweep plots,
- and a final monotonicity specification table for later experiments.

In [ ]:
# Imports and notebook setup

from pathlib import Path
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch

from IPython.display import display
#from sklearn.neighbors import NearestNeighbors

from src.models import MLPRegressor
from src.training import DEFAULT_TRAIN_CONFIG, make_standard_experiment_data

from src.monotonicity import (
    predict_numpy,
    make_split_audit_df,
    summarize_pairs,
    build_local_pairs,
    compute_pairwise_monotonicity_summary,
    compute_gradient_sign_audit,
    select_representative_rows,
    make_counterfactual_sweep_df,
    save_chosen_pair_artifacts,
    save_training_spec,
    load_chosen_pair_artifacts,
    load_training_spec,
)

sns.set_theme(style="whitegrid", context="talk")
pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 180)

PROJECT_ROOT = Path("..").resolve()
RESULTS_ROOT = PROJECT_ROOT / "results"
device = "cuda" if torch.cuda.is_available() else "cpu"

print("Project root:", PROJECT_ROOT)
print("Results root:", RESULTS_ROOT)
print("Using device:", device)

## Candidate monotonicity hypotheses

The initial candidate assumptions are:

| Feature | Direction | Initial status | Rationale |
|---|---:|---|---|
| `tmin` | increasing | candidate | warmer recent conditions plausibly increase next-day maximum temperature |
| `tavg` | increasing | candidate | same |
| `tmax` | increasing | candidate | same |
| `prcp` | decreasing | exploratory | plausible in some regimes, but risky as a global constraint |
| `dayofyear_sin` | none | exclude | cyclical feature; global monotonicity is not meaningful |
| `dayofyear_cos` | none | exclude | cyclical feature; global monotonicity is not meaningful |
| `rand_noise` | none | exclude | negative control |

These are **not final pseudo-labels yet**. They are hypotheses to audit.

In [ ]:
# Load the standard dataset, baseline model, and build a single audit dataframe

config = DEFAULT_TRAIN_CONFIG
data = make_standard_experiment_data(PROJECT_ROOT, config)

feature_cols = data.feature_cols
target_col = "tmax_next"

print("Train/val/test sizes:", len(data.train_df), len(data.val_df), len(data.test_df))
print("Feature columns:", feature_cols)

BASELINE_MODEL_PATH = RESULTS_ROOT / "baseline" / "best_model.pt"
assert BASELINE_MODEL_PATH.exists(), (
    f"Baseline checkpoint not found at {BASELINE_MODEL_PATH}. "
    "Run 03_baseline_training.ipynb first."
)

baseline_model = MLPRegressor(
    input_dim=len(feature_cols),
    hidden_dims=(64, 64),
    dropout=0.0,
)
baseline_model.load_state_dict(torch.load(BASELINE_MODEL_PATH, map_location=device))
baseline_model.to(device).eval()

pred_train = predict_numpy(baseline_model, data.X_train, device=device)
pred_val   = predict_numpy(baseline_model, data.X_val,   device=device)
pred_test  = predict_numpy(baseline_model, data.X_test,  device=device)

audit_train = make_split_audit_df(
    data.train_df, data.X_train, pred_train, "train", feature_cols, target_col=target_col
)
audit_val = make_split_audit_df(
    data.val_df, data.X_val, pred_val, "val", feature_cols, target_col=target_col
)
audit_test = make_split_audit_df(
    data.test_df, data.X_test, pred_test, "test", feature_cols, target_col=target_col
)

audit_df = pd.concat([audit_train, audit_val, audit_test], ignore_index=True)
audit_df["row_id"] = np.arange(len(audit_df))

# Rebuild split-level dataframes from audit_df so they carry the global row_id
audit_train = audit_df.loc[audit_df["split"] == "train"].copy().reset_index(drop=True)
audit_val   = audit_df.loc[audit_df["split"] == "val"].copy().reset_index(drop=True)
audit_test  = audit_df.loc[audit_df["split"] == "test"].copy().reset_index(drop=True)

split_arrays = {
    "train": data.X_train,
    "val": data.X_val,
    "test": data.X_test,
}

split_dfs = {
    "train": audit_train,
    "val": audit_val,
    "test": audit_test,
}

print("Audit dataframe shape:", audit_df.shape)
display(audit_df.head())

In [ ]:
# Encode monotonicity candidates and pair-construction settings

constraint_specs = [
    {
        "feature": "tmin",
        "sign": +1,
        "direction": "nondecreasing",
        "status": "candidate",
        "priority": "high",
        "rationale": "Warmer recent minimum temperature should usually not reduce next-day tmax locally.",
    },
    {
        "feature": "tavg",
        "sign": +1,
        "direction": "nondecreasing",
        "status": "candidate",
        "priority": "high",
        "rationale": "Warmer recent average temperature should usually not reduce next-day tmax locally.",
    },
    {
        "feature": "tmax",
        "sign": +1,
        "direction": "nondecreasing",
        "status": "candidate",
        "priority": "high",
        "rationale": "Warmer recent maximum temperature should usually not reduce next-day tmax locally.",
    },
    {
        "feature": "prcp",
        "sign": -1,
        "direction": "nonincreasing",
        "status": "exploratory",
        "priority": "low",
        "rationale": "More precipitation may be associated with lower next-day tmax, but the relation may be regime-dependent.",
    },
    {
        "feature": "dayofyear_sin",
        "sign": np.nan,
        "direction": "none",
        "status": "exclude",
        "priority": "none",
        "rationale": "Cyclical season encoding is not globally monotone.",
    },
    {
        "feature": "dayofyear_cos",
        "sign": np.nan,
        "direction": "none",
        "status": "exclude",
        "priority": "none",
        "rationale": "Cyclical season encoding is not globally monotone.",
    },
    {
        "feature": "rand_noise",
        "sign": np.nan,
        "direction": "none",
        "status": "exclude",
        "priority": "none",
        "rationale": "Negative control: no monotonic prior should be imposed.",
    },
]

constraints_df = pd.DataFrame(constraint_specs)
display(constraints_df)

pair_setting_specs = [
    {
        "setting": "local_knn_loose",
        "k_neighbors": 25,
        "context_max_dist": 2.0,
        "min_feature_gap_scaled": 0.25,
        "max_dayofyear_gap_days": np.nan,
        "max_pairs_per_anchor": 10,
        "notes": "Loose local matching on non-constrained features.",
    },
    {
        "setting": "local_knn_medium",
        "k_neighbors": 20,
        "context_max_dist": 1.5,
        "min_feature_gap_scaled": 0.35,
        "max_dayofyear_gap_days": 45.0,
        "max_pairs_per_anchor": 8,
        "notes": "Moderate local matching with seasonal closeness.",
    },
    {
        "setting": "local_knn_strict",
        "k_neighbors": 15,
        "context_max_dist": 1.0,
        "min_feature_gap_scaled": 0.50,
        "max_dayofyear_gap_days": 30.0,
        "max_pairs_per_anchor": 5,
        "notes": "Strict local matching; fewer but cleaner pairs.",
    },
]

pair_settings_df = pd.DataFrame(pair_setting_specs)
display(pair_settings_df)

In [ ]:
# Monotonicity audit helper functions
from numpy.typing import ArrayLike

def circular_dayofyear_gap(day_a: ArrayLike, day_b: ArrayLike, period: float = 365.25) -> np.ndarray:
    a = np.asarray(day_a, dtype=float)
    b = np.asarray(day_b, dtype=float)
    diff = np.abs(a - b)
    return np.minimum(diff, period - diff)

print("Helper functions defined.")

In [ ]:
# Build pair sets for each constrained feature, each setting, and each split

candidate_constraints_df = constraints_df.query("status in ['candidate', 'exploratory']").copy()
pair_store: dict[tuple[str, str, str], pd.DataFrame] = {}
pair_diag_records: list[dict] = []

for split in ["train", "val", "test"]:
    print(f"\nBuilding local pairs for split={split}")
    for _, spec in candidate_constraints_df.iterrows():
        feature = spec["feature"]
        sign = int(spec["sign"])

        for _, setting_row in pair_settings_df.iterrows():
            setting = {str(k): v for k, v in setting_row.to_dict().items()}
            setting_name = str(setting["setting"])

            pairs_df = build_local_pairs(
                audit_df,
                feature=feature,
                sign=sign,
                setting=setting,
                feature_cols=feature_cols,
                split=split,
            )

            pair_store[(split, feature, setting_name)] = pairs_df
            pair_diag_records.append(
                summarize_pairs(
                    pairs_df,
                    split=split,
                    feature=feature,
                    setting=setting_name,
                )
            )

pair_diagnostics_df = pd.DataFrame(pair_diag_records)
display(pair_diagnostics_df.round(4))

In [ ]:
# Audit monotonicity on labels and on baseline predictions

label_records: list[dict] = []
baseline_records: list[dict] = []

constraint_lookup = constraints_df.set_index("feature").to_dict(orient="index")

for (split, feature, setting_name), pairs_df in pair_store.items():
    sign = int(constraint_lookup[feature]["sign"])

    label_records.append(
        compute_pairwise_monotonicity_summary(
            audit_df,
            pairs_df,
            value_col="y_true",
            split=split,
            feature=feature,
            setting=setting_name,
            sign=sign,
        )
    )

    baseline_records.append(
        compute_pairwise_monotonicity_summary(
            audit_df,
            pairs_df,
            value_col="baseline_pred",
            split=split,
            feature=feature,
            setting=setting_name,
            sign=sign,
        )
    )

label_audit_df = pd.DataFrame(label_records)
baseline_audit_df = pd.DataFrame(baseline_records)

print("Label-level audit:")
display(label_audit_df.round(4))

print("Baseline-prediction audit:")
display(baseline_audit_df.round(4))

In [ ]:
# Heatmaps for pair counts and violation rates

feature_order = candidate_constraints_df["feature"].tolist()
setting_order = pair_settings_df["setting"].tolist()

fig, axes = plt.subplots(1, 3, figsize=(20, 5))

pair_pivot = (
    pair_diagnostics_df.query("split == 'train'")
    .pivot(index="feature", columns="setting", values="n_pairs")
    .reindex(index=feature_order, columns=setting_order)
)
sns.heatmap(pair_pivot, annot=True, fmt=".0f", cmap="Blues", ax=axes[0])
axes[0].set_title("Train pair counts")
axes[0].set_xlabel("Pair setting")
axes[0].set_ylabel("Feature")

label_pivot = (
    label_audit_df.query("split == 'train'")
    .pivot(index="feature", columns="setting", values="violation_rate")
    .reindex(index=feature_order, columns=setting_order)
)
sns.heatmap(label_pivot, annot=True, fmt=".3f", cmap="viridis_r", vmin=0.0, vmax=1.0, ax=axes[1])
axes[1].set_title("Train label violation rate")
axes[1].set_xlabel("Pair setting")
axes[1].set_ylabel("Feature")

baseline_pivot = (
    baseline_audit_df.query("split == 'test'")
    .pivot(index="feature", columns="setting", values="violation_rate")
    .reindex(index=feature_order, columns=setting_order)
)
sns.heatmap(baseline_pivot, annot=True, fmt=".3f", cmap="viridis_r", vmin=0.0, vmax=1.0, ax=axes[2])
axes[2].set_title("Test baseline violation rate")
axes[2].set_xlabel("Pair setting")
axes[2].set_ylabel("Feature")

plt.tight_layout()
plt.show()

In [ ]:
# Choose a preferred pair-construction setting per feature based on train label audit

MIN_TRAIN_PAIRS_FOR_SELECTION = 150

train_label_df = label_audit_df.query("split == 'train'").copy()
train_label_df["eligible"] = train_label_df["n_pairs"] >= MIN_TRAIN_PAIRS_FOR_SELECTION

# Lower is better; ineligible settings are penalized heavily.
train_label_df["selection_score"] = train_label_df["violation_rate"] + np.where(
    train_label_df["eligible"], 0.0, 10.0
)

best_settings_df = (
    train_label_df
    .sort_values(["feature", "selection_score", "mean_hinge_violation", "setting"])
    .groupby("feature", as_index=False)
    .first()
    [
        [
            "feature",
            "setting",
            "n_pairs",
            "violation_rate",
            "mean_hinge_violation",
            "eligible",
        ]
    ]
    .rename(
        columns={
            "setting": "chosen_setting",
            "n_pairs": "chosen_train_n_pairs",
            "violation_rate": "train_label_violation_rate",
            "mean_hinge_violation": "train_label_mean_hinge",
            "eligible": "chosen_setting_meets_pair_threshold",
        }
    )
)

print("Preferred pair setting per feature:")
display(best_settings_df.round(4))

selected_test_label_df = (
    label_audit_df
    .merge(best_settings_df[["feature", "chosen_setting"]], left_on=["feature", "setting"], right_on=["feature", "chosen_setting"])
    .query("split == 'test'")
    [
        [
            "feature",
            "setting",
            "violation_rate",
            "mean_hinge_violation",
            "n_pairs",
        ]
    ]
    .rename(
        columns={
            "setting": "chosen_setting",
            "violation_rate": "test_label_violation_rate",
            "mean_hinge_violation": "test_label_mean_hinge",
            "n_pairs": "test_label_n_pairs",
        }
    )
)

selected_test_baseline_df = (
    baseline_audit_df
    .merge(best_settings_df[["feature", "chosen_setting"]], left_on=["feature", "setting"], right_on=["feature", "chosen_setting"])
    .query("split == 'test'")
    [
        [
            "feature",
            "setting",
            "violation_rate",
            "mean_hinge_violation",
            "n_pairs",
        ]
    ]
    .rename(
        columns={
            "setting": "chosen_setting",
            "violation_rate": "test_baseline_violation_rate",
            "mean_hinge_violation": "test_baseline_mean_hinge",
            "n_pairs": "test_baseline_n_pairs",
        }
    )
)

preferred_compare_df = (
    best_settings_df
    .merge(selected_test_label_df, on=["feature", "chosen_setting"], how="left")
    .merge(selected_test_baseline_df, on=["feature", "chosen_setting"], how="left")
)
preferred_compare_df["baseline_minus_label_violation_test"] = (
    preferred_compare_df["test_baseline_violation_rate"] - preferred_compare_df["test_label_violation_rate"]
)

print("Preferred-setting comparison:")
display(preferred_compare_df.round(4))

# Plot label vs baseline violation rates on the chosen settings
plot_df = preferred_compare_df.copy()

x = np.arange(len(plot_df))
w = 0.38

plt.figure(figsize=(10, 5))
plt.bar(x - w / 2, plot_df["test_label_violation_rate"], width=w, label="label violation rate (test)")
plt.bar(x + w / 2, plot_df["test_baseline_violation_rate"], width=w, label="baseline violation rate (test)")
plt.xticks(x, plot_df["feature"].astype(str).tolist(), rotation=20)
plt.ylabel("Violation rate")
plt.title("Chosen-setting monotonicity audit on test split")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Gradient-sign monotonicity audit on train and test

gradient_records: list[dict] = []

for split in ["train", "test"]:
    X_split = split_arrays[split]

    for _, spec in candidate_constraints_df.iterrows():
        gradient_records.append(
            compute_gradient_sign_audit(
                baseline_model,
                X_split,
                split=split,
                constrained_feature=spec["feature"],
                sign=int(spec["sign"]),
                feature_cols=feature_cols,
                device=device,
                batch_size=512,
            )
        )

gradient_audit_df = pd.DataFrame(gradient_records)

print("Gradient-sign audit:")
display(gradient_audit_df.round(4))

# Plot test gradient-sign violation rates
grad_test_plot_df = gradient_audit_df.query("split == 'test'").copy()

plt.figure(figsize=(8, 4))
sns.barplot(data=grad_test_plot_df, x="feature", y="grad_violation_rate", hue="feature", dodge=False)
plt.title("Test gradient-sign violation rate")
plt.xlabel("Feature")
plt.ylabel("Violation rate")
plt.legend([], [], frameon=False)
plt.tight_layout()
plt.show()

In [ ]:
# Local counterfactual sweeps on representative test rows

rep_rows = select_representative_rows(
    audit_test.reset_index(drop=True),
    col="y_true",
    quantiles=(0.1, 0.5, 0.9),
)

print("Representative test rows:")
display(
    rep_rows[
        ["row_id", "datetime", "season", "y_true", "baseline_pred", "tmin", "tavg", "tmax", "prcp"]
    ].round(3)
)

features_to_plot = candidate_constraints_df["feature"].tolist()

for feature_name in features_to_plot:
    grid_orig = np.linspace(
        audit_train[feature_name].quantile(0.05),
        audit_train[feature_name].quantile(0.95),
        60,
    )

    fig, axes = plt.subplots(1, len(rep_rows), figsize=(5 * len(rep_rows), 4), sharey=True)
    if len(rep_rows) == 1:
        axes = [axes]

    for ax, (_, base_row) in zip(axes, rep_rows.iterrows()):
        cf_df = make_counterfactual_sweep_df(
            baseline_model,
            base_row=base_row,
            feature_name=feature_name,
            grid_orig=grid_orig,
            feature_cols=feature_cols,
            scaler=data.scaler,
            device=device,
        )

        ax.plot(cf_df["feature_value_orig"], cf_df["prediction"], linewidth=2)
        ax.scatter(
            [cf_df["base_feature_value_orig"].iloc[0]],
            [cf_df["base_pred"].iloc[0]],
            color="red",
            zorder=5,
            label="original point",
        )
        ax.set_title(
            f"{feature_name} sweep\n"
            f"row_id={int(base_row['row_id'])}, y={base_row['y_true']:.1f}, pred={base_row['baseline_pred']:.1f}"
        )
        ax.set_xlabel(feature_name)
        ax.set_ylabel("Baseline prediction")
        ax.legend()

    plt.tight_layout()
    plt.show()

In [ ]:
# Build the final monotonicity specification table

test_grad_df = (
    gradient_audit_df.query("split == 'test'")
    [
        [
            "feature",
            "grad_violation_rate",
            "mean_signed_gradient",
            "median_signed_gradient",
        ]
    ]
    .rename(
        columns={
            "grad_violation_rate": "test_grad_violation_rate",
            "mean_signed_gradient": "test_mean_signed_gradient",
            "median_signed_gradient": "test_median_signed_gradient",
        }
    )
)

final_mono_spec_df = (
    constraints_df
    .merge(best_settings_df, on="feature", how="left")
    .merge(selected_test_label_df, on=["feature", "chosen_setting"], how="left")
    .merge(selected_test_baseline_df, on=["feature", "chosen_setting"], how="left")
    .merge(test_grad_df, on="feature", how="left")
)

MIN_TRAIN_PAIRS_FOR_KEEP = 150
MAX_TRAIN_LABEL_VIOLATION_FOR_KEEP = 0.35
MAX_TRAIN_LABEL_VIOLATION_FOR_MAYBE = 0.45

def suggest_keep(row: pd.Series) -> str:
    status = row["status"]

    if status == "exclude":
        return "no"

    if status == "exploratory":
        if (
            pd.notna(row["chosen_train_n_pairs"])
            and row["chosen_train_n_pairs"] >= MIN_TRAIN_PAIRS_FOR_KEEP
            and pd.notna(row["train_label_violation_rate"])
            and row["train_label_violation_rate"] <= 0.30
        ):
            return "maybe"
        return "no"

    # candidate
    if (
        pd.notna(row["chosen_train_n_pairs"])
        and row["chosen_train_n_pairs"] >= MIN_TRAIN_PAIRS_FOR_KEEP
        and pd.notna(row["train_label_violation_rate"])
        and row["train_label_violation_rate"] <= MAX_TRAIN_LABEL_VIOLATION_FOR_KEEP
    ):
        return "yes"

    if (
        pd.notna(row["chosen_train_n_pairs"])
        and row["chosen_train_n_pairs"] >= MIN_TRAIN_PAIRS_FOR_KEEP
        and pd.notna(row["train_label_violation_rate"])
        and row["train_label_violation_rate"] <= MAX_TRAIN_LABEL_VIOLATION_FOR_MAYBE
    ):
        return "maybe"

    return "no"

def confidence_from_label_violation(row: pd.Series) -> str:
    if row["status"] == "exclude":
        return "high"

    v = row["train_label_violation_rate"]
    if pd.isna(v):
        return "unknown"
    if v <= 0.25:
        return "high"
    if v <= 0.35:
        return "medium"
    return "low"

def make_notes(row: pd.Series) -> str:
    notes = []

    if row["feature"] in {"dayofyear_sin", "dayofyear_cos"}:
        notes.append("cyclical feature; do not impose global monotonicity")
    if row["feature"] == "rand_noise":
        notes.append("negative control; exclude")
    if row["feature"] == "prcp":
        notes.append("inspect regime dependence before global use")

    if pd.notna(row.get("baseline_minus_label_violation_test", np.nan)):
        gap = row["baseline_minus_label_violation_test"]
        if gap > 0.03:
            notes.append("baseline is less monotone than labels on test")
        elif gap < -0.03:
            notes.append("baseline is already more monotone than labels on test")
        else:
            notes.append("baseline and labels have similar test violation rates")

    if pd.notna(row.get("test_grad_violation_rate", np.nan)):
        if row["test_grad_violation_rate"] > 0.40:
            notes.append("local gradient violations remain frequent")
        elif row["test_grad_violation_rate"] < 0.20:
            notes.append("gradient-sign behaviour is relatively stable")

    return "; ".join(notes)

final_mono_spec_df["baseline_minus_label_violation_test"] = (
    final_mono_spec_df["test_baseline_violation_rate"] - final_mono_spec_df["test_label_violation_rate"]
)
final_mono_spec_df["suggested_keep_for_training"] = final_mono_spec_df.apply(suggest_keep, axis=1)
final_mono_spec_df["confidence"] = final_mono_spec_df.apply(confidence_from_label_violation, axis=1)
final_mono_spec_df["notes"] = final_mono_spec_df.apply(make_notes, axis=1)

final_mono_spec_df = final_mono_spec_df[
    [
        "feature",
        "direction",
        "sign",
        "status",
        "priority",
        "chosen_setting",
        "chosen_train_n_pairs",
        "train_label_violation_rate",
        "test_label_violation_rate",
        "test_baseline_violation_rate",
        "baseline_minus_label_violation_test",
        "test_grad_violation_rate",
        "suggested_keep_for_training",
        "confidence",
        "rationale",
        "notes",
    ]
]

print("Final monotonicity specification:")
display(final_mono_spec_df.round(4))

In [ ]:
# Save audit outputs

MONO_AUDIT_DIR = RESULTS_ROOT / "monotonicity_audit"
MONO_AUDIT_DIR.mkdir(parents=True, exist_ok=True)

pair_diagnostics_df.to_csv(MONO_AUDIT_DIR / "pair_diagnostics.csv", index=False)
label_audit_df.to_csv(MONO_AUDIT_DIR / "label_monotonicity_audit.csv", index=False)
baseline_audit_df.to_csv(MONO_AUDIT_DIR / "baseline_monotonicity_audit.csv", index=False)
gradient_audit_df.to_csv(MONO_AUDIT_DIR / "gradient_sign_audit.csv", index=False)
best_settings_df.to_csv(MONO_AUDIT_DIR / "best_pair_settings.csv", index=False)
preferred_compare_df.to_csv(MONO_AUDIT_DIR / "preferred_setting_comparison.csv", index=False)
final_mono_spec_df.to_csv(MONO_AUDIT_DIR / "final_monotonicity_spec.csv", index=False)

audit_config = {
    "feature_cols": feature_cols,
    "candidate_features": candidate_constraints_df["feature"].tolist(),
    "pair_settings": pair_settings_df.to_dict(orient="records"),
    "min_train_pairs_for_selection": int(MIN_TRAIN_PAIRS_FOR_SELECTION),
    "min_train_pairs_for_keep": int(MIN_TRAIN_PAIRS_FOR_KEEP),
    "max_train_label_violation_for_keep": float(MAX_TRAIN_LABEL_VIOLATION_FOR_KEEP),
    "max_train_label_violation_for_maybe": float(MAX_TRAIN_LABEL_VIOLATION_FOR_MAYBE),
    "baseline_model_path": str(BASELINE_MODEL_PATH),
}

with open(MONO_AUDIT_DIR / "audit_config.json", "w") as fh:
    json.dump(audit_config, fh, indent=2)

print("Saved monotonicity audit outputs to:", MONO_AUDIT_DIR)

## Next step

Once the final monotonicity specification is accepted, the next notebook should be:

# `08_rq2_monotonicity_training.ipynb`

That notebook should:

1. implement the monotonicity regularization loss,
2. use the selected constrained features from this notebook,
3. use the selected pair-construction rule from this notebook,
4. compare baseline ERM against monotonicity-regularized training,
5. sweep the monotonicity regularization strength,
6. and report both predictive performance and monotonicity violation metrics.

In [ ]:
# Save chosen pair artifacts and compact training specs for notebook 08

MONO_AUDIT_DIR = RESULTS_ROOT / "monotonicity_audit"
MONO_AUDIT_DIR.mkdir(parents=True, exist_ok=True)

chosen_pairs_manifest_df = save_chosen_pair_artifacts(
    MONO_AUDIT_DIR,
    pair_store=pair_store,
    best_settings_df=best_settings_df,
    split_dfs=split_dfs,
    constraints_df=constraints_df,
)

training_spec_main_df = save_training_spec(
    final_mono_spec_df,
    MONO_AUDIT_DIR,
    keep_values=("yes",),
    name="training_spec_main",
)

training_spec_with_optional_df = save_training_spec(
    final_mono_spec_df,
    MONO_AUDIT_DIR,
    keep_values=("yes", "maybe"),
    name="training_spec_with_optional",
)

print("Chosen pair artifacts manifest:")
display(chosen_pairs_manifest_df.round(4))

print("Notebook 08 main training spec:")
display(training_spec_main_df.round(4))

print("Notebook 08 optional training spec:")
display(training_spec_with_optional_df.round(4))

In [ ]:
# Dry-run load of saved monotonicity artifacts

loaded_manifest_df, loaded_pair_idx_store = load_chosen_pair_artifacts(MONO_AUDIT_DIR)
loaded_training_spec_df = load_training_spec(MONO_AUDIT_DIR, name="training_spec_main")

print("Loaded manifest:")
display(loaded_manifest_df.head())

print("Loaded training spec:")
display(loaded_training_spec_df)

print("Available pair-index keys:")
print(sorted(loaded_pair_idx_store.keys())[:10])